# SCBE Code A/B/C Matched-Budget Benchmark (Kaggle-safe)

**3-way comparison** of code training data richness:

| Condition | Data | What the model sees |
|-----------|------|---------------------|
| **A** (baseline) | `round5_code_baseline_l3.jsonl` | L3 only — surface code |
| **B** (multiview) | `round5_code_multiview_l0l3.jsonl` | L0-L3 — structural multi-view |
| **C** (canonical) | `round6_canonical_ir_l3.jsonl` | L-1 + L0-L3 — canonical IR (code notes + graph + spectral + hyperbolic) |

All three train on the **same model, same steps, same token budget**. Only data richness differs.

Expected path:
- Kaggle GPU (`T4` or `P100`) enabled
- Internet on
- Run all
- Result packet written to `/kaggle/working/code_abc_matched_budget_summary.json`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.44', 'datasets', 'accelerate>=0.33', 'peft>=0.12', 'bitsandbytes>=0.43'], check=True)


In [ ]:
import os, subprocess
REPO_DIR = '/kaggle/working/SCBE-AETHERMOORE'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/issdandavis/SCBE-AETHERMOORE.git', REPO_DIR], check=True)
%cd /kaggle/working/SCBE-AETHERMOORE


In [ ]:
# Run the 3-way A/B/C benchmark (uses 2-way as fallback if round6 missing)
import os
abc_script = 'scripts/research/train_code_abc_kaggle.py'
ab_script = 'scripts/research/train_code_ab_kaggle_safe.py'

if os.path.exists(abc_script):
    !python {abc_script}
else:
    print("Round6 canonical dataset not found, falling back to 2-way A/B")
    !python {ab_script}

In [ ]:
import json
from pathlib import Path

# Try ABC first, fall back to AB
for name in ['code_abc_matched_budget_summary.json', 'code_ab_matched_budget_summary.json']:
    p = Path('/kaggle/working') / name
    if p.exists():
        summary = json.loads(p.read_text(encoding='utf-8'))
        print(json.dumps(summary, indent=2))
        break
else:
    print("No summary file found")

For CPU smoke test: rerun with `--allow-cpu-smoke`. That mode verifies the pipeline contract, not benchmark claims.

```
!python scripts/research/train_code_abc_kaggle.py --allow-cpu-smoke
```